# 5.0 Deploy: batch transform on the production split (run in SageMaker Studio)

Deployment is batch transform only (no real-time endpoint, nothing left running). Uses the exact package ARN recorded by 4.2 in `xgboost_eval.json`, which must be Approved. Input/output join is enabled (output rows = input features + class probabilities) so Part D can run batch-mode monitoring on production outputs. Production labels are not used for training; `true_activity_id` is carried through only for later quality monitoring.

In [ ]:
# SageMaker Python SDK v3 (default on current Studio images) removed the v2 modules this notebook uses.
# Pin v2; on first run this installs it and restarts the kernel - then run the notebook again from the top.
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json
import boto3, numpy as np, pandas as pd
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_parquet(name):
    b, k = split_key(SPLITS + name)
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
def read_json(uri):
    b, k = split_key(uri)
    return json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
CLASSES = list(range(K))
def class_name(i):
    return ACTIVITY_NAMES[LABEL_MAP[int(i)]]
print(K, "classes;", len(FEATURES), "features")


In [ ]:
import sagemaker
from src.pamap2.features import intensity_of
sm = session.client("sagemaker")
sm_session = sagemaker.Session(boto_session=session)
import time
from sagemaker.transformer import Transformer

def upload_features_csv(df, key):
    """Header-less, feature-only CSV in feature_columns.json order."""
    E.check_feature_columns(df, FEATURES)
    body = df[FEATURES].to_csv(header=False, index=False).encode()
    uri = s3_uri(cfg, "models", key)
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=body)
    return uri

def run_transform(model_name, input_uri, output_prefix, join=False):
    """Batch transform on ml.m5.large x1; waits for completion. Nothing stays running afterwards."""
    t = Transformer(model_name=model_name, instance_count=1, instance_type="ml.m5.large",
                    strategy="MultiRecord", assemble_with="Line", accept="text/csv",
                    output_path=output_prefix, sagemaker_session=sm_session)
    t.transform(data=input_uri, content_type="text/csv", split_type="Line",
                join_source="Input" if join else "None", wait=True, logs=False)
    return output_prefix + input_uri.rsplit("/", 1)[1] + ".out"

def read_probs(out_uri, k):
    b, kk = split_key(out_uri)
    return E.parse_probs(s3.get_object(Bucket=b, Key=kk)["Body"].read(), k)


## Create model from the latest approved package

In [ ]:
pkg_arn = read_json(REPORTS + "xgboost_eval.json")["model_package_arn"]   # exactly the package 4.2 evaluated
status = sm.describe_model_package(ModelPackageName=pkg_arn)["ModelApprovalStatus"]
assert status == "Approved", f"{pkg_arn} is {status}; run the manual approval cell at the end of 4.2 first."
model_name = f"{cfg['prefix']}-prod-{int(time.time())}"
sm.create_model(ModelName=model_name, ExecutionRoleArn=cfg["role_arn"],
                Containers=[{"ModelPackageName": pkg_arn}])
prod = read_parquet("production.parquet").reset_index(drop=True)
prod_uri = upload_features_csv(prod, "transform-input/production/features.csv")
print("Package:", pkg_arn, "|", len(prod), "production rows")

## Batch transform with input/output join (billable: one ml.m5.large job)

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
try:
    out_uri = run_transform(model_name, prod_uri, s3_uri(cfg, "models", f"transform-output/production/{stamp}/"), join=True)
    probs = read_probs(out_uri, K)
    assert len(probs) == len(prod), (len(probs), len(prod))
finally:
    sm.delete_model(ModelName=model_name)   # only the temporary model created above
print("Joined output:", out_uri)

## Write predictions

In [ ]:
cls = E.probs_to_labels(probs, K)
act = [LABEL_MAP[int(c)] for c in cls]
preds = pd.DataFrame({
    "window_id": prod["window_id"].astype(str), "subject_id": prod["subject_id"],
    "predicted_class": cls, "predicted_activity_id": act,
    "activity_name": [ACTIVITY_NAMES.get(a) for a in act],
    "intensity": [intensity_of(a) for a in act],
    "confidence": probs.max(axis=1),
    "true_activity_id": prod["activity_id"],
    "model_package_arn": pkg_arn,
})
assert len(preds) == len(prod) and preds["activity_name"].notna().all()
buf = io.BytesIO(); preds.to_parquet(buf, index=False)
b, k = split_key(s3_uri(cfg, "models", f"predictions/production/predictions-{stamp}.parquet"))
s3.put_object(Bucket=b, Key=k, Body=buf.getvalue())
print(f"Wrote s3://{b}/{k}")
print(preds["activity_name"].value_counts().to_string())
print(preds["intensity"].value_counts().to_string())

## Confirm nothing is left running

In [ ]:
print("Endpoints:", [e["EndpointName"] for e in sm.list_endpoints()["Endpoints"]])
print("Transform jobs in progress:", len(sm.list_transform_jobs(StatusEquals="InProgress")["TransformJobSummaries"]))